# MHD equilibrium surrogates: results

The stored equilibrium surrogates on the test split of `equilibrium_a_3k_spline_input`. Three
entries throughout: the trilinear equilibrium interpolator, the data-only MLP, and the MLP
whose second stage minimized the force residual alone. Nothing is trained or scored here: the
notebook reads the models in `mhd_equilibrium_forward_models/trained_models/`, the run
directories under `models_training/`, and the tables written by `run_test_set_evaluation.py`.

1. **Training curves**, per model and per stage.
2. **Test-set errors** and force residual.
3. **One model in detail**: the force-trained MLP beside the DESC solve, for two equilibria
   and binned by beta.
4. **The interpolator**: exact on its lattice, saturating off it.

Every length, pressure and field is reported at the machine scale
(`diagnostic_design_point.MACHINE_SCALE`, `R0 = 5.5 m` and `B = 9 T`, the scale the
inference runs at), carried from the dataset's normalized units by ideal-MHD scale
invariance; nothing is re-solved or re-scored. Section 4 alone reads the
interpolator's lattice as stored.

Figures go to `plots/` and tables to `tables/`.

In [ ]:
import json
import pathlib

import desc.compute
import desc.grid
import desc.vmec_utils
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mpl_colors
from matplotlib import ticker as mpl_ticker
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    diagnostic_design_point,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    desc_types,
    desc_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    plot as ideal_mhd_plot,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
    interpolator_utils,
    model_storage_utils,
    preprocessing_types,
    surrogate_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_mhd_equilibrium_forward_models as plots,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    pubfig,
)

# Anchor the output directory to the package location, not the kernel's cwd, so the
# notebook runs from any working directory.
ROOT = pathlib.Path(plots.__file__).parents[2]
OUTPUT_DIR = (
    ROOT
    / "results_plots_tables"
    / "results"
    / "mhd_equilibrium_forward_models"
    / "test_set_evaluation"
)
assert OUTPUT_DIR.is_dir(), OUTPUT_DIR
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)
MODELS_DIR = ROOT / "mhd_equilibrium_forward_models" / "trained_models"
DATA_DIR = ROOT.parents[1] / "data"
DATASET = dataset_utils.DEFAULT_DATASET
# Written by run_test_set_evaluation.py, not by this notebook.
METRICS_CSV = TABLES_DIR / "mhd_surrogate_test_set_metrics.csv"
PER_SAMPLE_CSV = TABLES_DIR / "mhd_surrogate_test_set_per_sample.csv"

# The model section 3 reads in detail, with the force residual alone in its second
# stage.
FOCUS_MODEL = "20260907_081512_mlp_data_fnorm_only_pca_hand_tuned"

# The machine every length, pressure and field below is reported at. The campaign
# and the dataset are in normalized units, which ideal-MHD scale invariance carries
# over exactly, so nothing is re-solved or re-scored.
MACHINE_SCALE = diagnostic_design_point.MACHINE_SCALE

# What a legend calls the force-trained network's prediction, and the solve it is read
# against.
PREDICTION_LABEL = "PINN"
SOLVE_LABEL = "DESC solve"

# What the figures call each entry; a model not listed here is labelled by its stored
# name. Edit the right-hand side to change the wording the thesis sees.
MODEL_LABELS = {
    "equilibrium_interpolator": "Tri-linear interpolator",
    "20260908_141134_mlp_data_pca_hand_tuned": "MLP",
    FOCUS_MODEL: PREDICTION_LABEL,
}

# Every figure and table lists the models in this order, matched against the end of a
# stored name; a configuration not listed sorts last.
MODEL_ORDER = (
    "equilibrium_interpolator",
    "mlp_data_pca_hand_tuned",
    "mlp_data_fnorm_only_pca_hand_tuned",
)


def save_pdf(figure, name: str) -> None:
    figure.savefig(PLOTS_DIR / f"{name}.pdf")


# The thesis prints the error chart and the force-residual chart as panels of one
# figure, so they share a figure size and an axes box fixed in inches.
# Constrained layout would shrink each box by its own labels instead.
PANEL_SIZE_IN = (3.5, 3.0)
PANEL_AXES_IN = (0.8, 0.8, 2.6, 1.95)  # left, bottom, width, height


def panel_figure():
    width, height = PANEL_SIZE_IN
    left, bottom, axes_width, axes_height = PANEL_AXES_IN
    figure = plt.figure(figsize=PANEL_SIZE_IN)
    panel = figure.add_axes(
        (left / width, bottom / height, axes_width / width, axes_height / height)
    )
    return figure, panel


def save_panel_pdf(figure, name: str) -> None:
    # Uncropped: the house style's tight bounding box would trim each panel by its own
    # labels and give the three different page sizes again. Passing
    # is not enough, since it falls back to that style.
    with plt.rc_context({"savefig.bbox": "standard"}):
        figure.savefig(PLOTS_DIR / f"{name}.pdf")


pubfig.apply_style()
registry_rows = []
for path in sorted(MODELS_DIR.glob("*.json")):
    stored = json.loads(path.read_text())
    training = stored.get("training", {})
    provenance = stored.get("provenance", {})
    registry_rows.append(
        {
            "model": stored["name"],
            "configuration": stored.get("configuration", stored["name"]),
            # The interpolator predates the description and carries `notes` instead.
            "description": stored.get("description") or stored.get("notes", ""),
            "kind": stored["kind"],
            "generated": stored.get("generated", ""),
            "n_parameters": training.get("n_parameters"),
            "n_latent": training.get("n_latent"),
            "n_epochs": training.get("n_epochs"),
            "train_mse": training.get("train_mse"),
            "validation_total": training.get("validation_total"),
            "train_seconds": training.get("train_seconds"),
            "dataset": provenance.get("dataset", ""),
            "run_directory": provenance.get("run_directory", ""),
        }
    )
# Sorted by name first, so two runs of one configuration keep their timestamp order.
registry = pd.DataFrame(registry_rows).sort_values("model", kind="stable")
model_ranks = [
    next(
        (rank for rank, ending in enumerate(MODEL_ORDER) if model.endswith(ending)),
        len(MODEL_ORDER),
    )
    for model in registry["model"]
]
registry = registry.iloc[np.argsort(model_ranks, kind="stable")].reset_index(drop=True)
registry

## 1. Training curves

The data term against epoch (dashed: validation), the only term every stage of every model
shares. It does not select the model: each stage selects on its own objective, drawn stage by
stage in the next figure.

In [ ]:
models = registry[registry["kind"] == model_storage_utils.MLP_MODEL_KIND]
assert not models.empty, f"no trained MLP in {MODELS_DIR}; run train_surrogate.py first"
histories = {}
for row in models.itertuples():
    run_directory = ROOT / row.run_directory
    assert (run_directory / "history.csv").exists(), (
        f"{row.model} points at {run_directory}, which holds no history.csv"
    )
    histories[row.model] = pd.read_csv(run_directory / "history.csv")
figure, axes = pubfig.subplots(width="double", aspect=0.45)
panel = axes[0]
for index, (name, history) in enumerate(histories.items()):
    colour = pubfig.CATEGORICAL_PALETTE[index % len(pubfig.CATEGORICAL_PALETTE)]
    # Computer Modern has no underscore glyph, so a stored name prints with spaces.
    label = MODEL_LABELS.get(name, name.replace("_", " "))
    panel.plot(history["epoch"], history["train_data"], color=colour, label=label)
    panel.plot(
        history["epoch"], history["validation_data"], color=colour, ls="--", lw=0.8
    )
panel.set_yscale("log")
panel.set_xlabel("epoch")
panel.set_ylabel("mean squared error")
# Under the axes in two columns: the stored names are long enough that a legend beside
# the axes leaves them an inch wide.
figure.legend(
    handles=panel.get_legend_handles_labels()[0],
    title="training (solid), validation (dashed)",
    ncol=2,
    loc="outside lower center",
)
save_pdf(figure, "mhd_surrogate_training_curves")
print({name: len(history) for name, history in histories.items()})

Each multi-stage run on its own, one panel per stage on its own epoch count (dashed:
validation). A stage starts from the parameters its predecessor selected, marked by the
circle, not from its last epoch. The force term is drawn as the residual itself.

In [ ]:
staged = [name for name, history in histories.items() if history["stage"].nunique() > 1]
run_directories = dict(zip(models["model"], models["run_directory"]))
for name in staged:
    history = histories[name]
    stages = json.loads(
        (ROOT / run_directories[name] / "summary.json").read_text()
    )["stages"]
    # One panel per stage, each counting its own epochs: a stage starts from the
    # parameters its predecessor selected, not from its last epoch, so a shared epoch
    # axis would draw a continuity the training does not have.
    figure, axes = pubfig.subplots(1, len(stages), width=6.8, aspect=0.75)
    for number, (panel, stage) in enumerate(zip(axes, stages), start=1):
        rows = history[history["stage"] == stage["label"]]
        epoch = rows["epoch"] - rows["epoch"].min()
        # A term a stage does not carry is recorded as exactly zero.
        carries_data = (rows["train_data"] != 0.0).any()
        carries_force = (rows["train_fnorm"] != 0.0).any()
        assert carries_data != carries_force, (
            f"stage {stage['label']!r} carries both terms or neither, which one axis "
            "cannot show"
        )
        if carries_force:
            # The term is `weight * log(residual)`, drawn as the residual itself.
            weight = float(stage["fnorm_weight"])
            train = np.exp(rows["train_fnorm"] / weight)
            validation = np.exp(rows["validation_fnorm"] / weight)
            colour = pubfig.PALETTE[2]
            panel.set_ylabel(r"force residual $\tilde{f}_{\mathrm{norm}}$")
        else:
            train = rows["train_data"]
            validation = rows["validation_data"]
            colour = pubfig.PALETTE[0]
            panel.set_ylabel("data loss")
        panel.plot(epoch, train, color=colour, label="training")
        panel.plot(epoch, validation, color=colour, ls="--", lw=0.8, label="validation")
        # The epoch the stage keeps, on the validation term it selects on.
        selected = rows[f"validation_{stage['selection_term']}"].idxmin()
        assert np.isclose(
            rows.loc[selected, "validation_total"], stage["validation_total"]
        ), f"stage {stage['label']!r}: the history disagrees with summary.json"
        panel.axvline(epoch[selected], color="0.5", ls=":", lw=0.8)
        panel.plot(
            epoch[selected],
            validation[selected],
            "o",
            mfc="none",
            mec="k",
            ms=6.0,
            label=(
                f"start of stage {number + 1}"
                if number < len(stages)
                else "final model"
            ),
        )
        panel.set_yscale("log")
        # Labels at 1, 2 and 5 of each decade: each term spans about one decade, which
        # the default labels mark once.
        panel.yaxis.set_major_locator(mpl_ticker.LogLocator(subs=(1.0, 2.0, 5.0)))
        panel.yaxis.set_major_formatter(
            mpl_ticker.LogFormatterSciNotation(
                labelOnlyBase=False, minor_thresholds=(3.0, 3.0)
            )
        )
        panel.set_xlabel(f"epoch of stage {number}")
        panel.legend()
        print(f"{name} stage {number}: selected epoch {epoch[selected]} of {len(rows)}")
    save_pdf(figure, f"mhd_surrogate_stages_{name}")
print("multi-stage runs:", staged or "none")

## 2. Test-set errors

The test split, scored by the evaluation campaign. The interpolator is the floor the networks
are read against; the data-only MLP (`mlp_data_pca_hand_tuned`) and the force-trained MLP
(`mlp_data_fnorm_only_pca_hand_tuned`) differ only in the force stage. The asserts check that
all rows come from one campaign on one test set. The error chart leaves out the toroidal flux,
which is the same in every dataset row; `psi_relative_error` is in the table.
The campaign scores in the dataset's normalized units; the two length errors (flux
surface and axis) are carried to the machine by `L`, every other metric being
dimensionless and therefore the same at both scales.

In [ ]:
assert METRICS_CSV.exists(), (
    f"{METRICS_CSV} is missing; run run_test_set_evaluation.py first"
)
metrics = pd.read_csv(METRICS_CSV).sort_values("model", kind="stable")
model_ranks = [
    next(
        (rank for rank, ending in enumerate(MODEL_ORDER) if model.endswith(ending)),
        len(MODEL_ORDER),
    )
    for model in metrics["model"]
]
metrics = metrics.iloc[np.argsort(model_ranks, kind="stable")].reset_index(drop=True)
# A length is `L` times longer at the machine; nothing else in the table has a unit.
LENGTH_COLUMNS = [
    "flux_surface_error_m",
    "magnetic_axis_error_m",
    "flux_surface_error_m_worst",
]
metrics[LENGTH_COLUMNS] *= MACHINE_SCALE.length_factor
assert metrics["n_samples"].nunique() == 1, (
    "the models were scored on different test sets"
)
assert metrics["settings_digest"].nunique() == 1, (
    "these rows come from different campaigns; "
    "rerun the evaluation for all models at once"
)
# Every model recomputes the target side of each metric, so the target force residual
# is a property of the test set alone and must agree across rows.
assert np.allclose(
    metrics["fnorm_target_p50"], metrics["fnorm_target_p50"].iloc[0], rtol=1e-6
), "the target force residual differs between models, so they scored different targets"

# Both bar charts leave out the data-only MLP without PCA: its force residual and
# toroidal-flux error are orders of magnitude above every other model.
bar_rows = metrics[~metrics["model"].str.endswith("mlp_data")].reset_index(drop=True)
bar_labels = [
    MODEL_LABELS.get(name, name.replace("_", " ")) for name in bar_rows["model"]
]
bar_colours = [
    pubfig.CATEGORICAL_PALETTE[index % len(pubfig.CATEGORICAL_PALETTE)]
    for index in range(len(bar_rows))
]
bar_width = 0.8 / max(len(bar_rows), 1)

# The toroidal flux is left out: it is the same in every row, so its relative error
# measures a constant while setting the range of the whole log axis.
ERROR_METRICS = (
    ("flux_surface_error_m", "flux surface [m]"),
    ("magnetic_axis_error_m", "axis [m]"),
    ("iota_computed_rmse", r"$\iota$ recomputed"),
    ("iota_predicted_rmse", r"$\iota$ predicted"),
    ("field_strength_relative_rmse", r"$|B|$ relative"),
)
present = [(name, label) for name, label in ERROR_METRICS if name in bar_rows.columns]
positions = np.arange(len(present), dtype=float)
figure, panel = panel_figure()
for index, row in bar_rows.iterrows():
    panel.bar(
        positions + index * bar_width,
        [max(float(row[name]), 0.0) for name, _ in present],
        width=bar_width,
        color=bar_colours[index],
        label=bar_labels[index],
    )
panel.set_xticks(positions + 0.4 - bar_width / 2.0)
# Tilted: five labels side by side overlap at single-column width.
panel.set_xticklabels(
    [label for _, label in present], rotation=30, ha="right", rotation_mode="anchor"
)
panel.set_yscale("log")
panel.set_ylabel("test-set error")
# Inside the axes, so the panel keeps the single-column width of the training curves
# it is printed beside; the headroom above the bars is the legend's.
panel.set_ylim(top=panel.get_ylim()[1] * 4.0)
panel.legend(loc="upper left", frameon=False)
save_panel_pdf(figure, "mhd_surrogate_error_metrics")
metrics

### Force balance on the test set

DESC's volume-averaged normalized force residual of the predicted equilibria: mean, median and
90th percentile over the test set (the training set was filtered at 1e-2). Every mean sits
above its median, so a few badly predicted samples carry the tail. Within a group, left to
right: the interpolator's three-parameter family, a network fitted to the coefficients, and
the same network after the force stage. The table holds the same numbers.

In [ ]:
FORCE_RESIDUAL_METRICS = (
    ("fnorm_predicted", "mean"),
    ("fnorm_predicted_p50", "median"),
    ("fnorm_predicted_p90", "90th percentile"),
)
positions = np.arange(len(FORCE_RESIDUAL_METRICS), dtype=float)
figure, panel = panel_figure()
for index, row in bar_rows.iterrows():
    panel.bar(
        positions + index * bar_width,
        [max(float(row[column]), 0.0) for column, _ in FORCE_RESIDUAL_METRICS],
        width=bar_width,
        color=bar_colours[index],
        label=bar_labels[index],
    )
panel.set_xticks(positions + 0.4 - bar_width / 2.0)
panel.set_xticklabels([label for _, label in FORCE_RESIDUAL_METRICS])
panel.set_yscale("log")
panel.set_ylabel("force residual")
# Down to 1e-3, so the axis carries a third decade tick. Autoscaling stops just
# under the shortest bar, leaving the reader two labelled decades for five bars.
panel.set_ylim(bottom=1e-3)
# No legend: the thesis prints this panel beside the error chart, whose legend gives
# the same colours.
save_panel_pdf(figure, "mhd_surrogate_force_residual")
metrics[["model", "fnorm_predicted", "fnorm_predicted_p50", "fnorm_predicted_p90"]]

## 3. One model in detail

The force-trained MLP beside the DESC solve each equilibrium came from: three test equilibria
in 3.1, the whole test split binned by beta in 3.2.

### 3.1 Equilibrium and profile prediction

Three test rows, chosen by their volume-averaged beta:

- **lowest beta**: the lowest-beta row (the split holds no zero-pressure row).
- **intermediate beta**: among the rows inside the split's interquartile range of beta, the
  one whose predicted transform has the lowest RMSE against its solve.
- **highest beta**: the highest-beta row whose prediction has a normalized force residual
  below 1e-2. Every DESC solve in the split is below that already, so the cut acts on the
  prediction: this case shows the model at high beta where it holds force balance, not at
  its worst.

The pressure is the model's input, the stored spline at its 99 knots.
Prediction and solve are both carried to the machine scale: the surrogate through
`load_surrogate(..., machine_scale=)`, the stored `R`, `Z` and `Psi` through
`surrogate_utils.flat_target_scale_factors`, and the pressure through `b^2`, as
`read_desc_truth` carries a truth.

In [ ]:
metadata = dataset_utils.load_metadata(DATA_DIR / f"{DATASET}_settings.json")
dataset = dataset_utils.load_dataset(DATA_DIR / f"{DATASET}.parquet", metadata)
split = dataset_utils.split_indices(dataset, metadata)
test_rows = np.asarray(split.test)
beta_percent = 100.0 * np.asarray(dataset.beta_vol)
solve_residual = np.asarray(dataset.fnorm_vol)

test_beta = beta_percent[test_rows]
# The intermediate case: among the rows inside the split's interquartile range of beta,
# the one whose predicted transform is closest to its solve.
lower_quartile, upper_quartile = np.percentile(test_beta, [25.0, 75.0])
intermediate_rows = test_rows[
    (test_beta >= lower_quartile) & (test_beta <= upper_quartile)
]
focus_per_sample = (
    pd.read_csv(PER_SAMPLE_CSV)
    .query("model == @FOCUS_MODEL")
    .set_index("equilibrium_id")
)
intermediate_iota_error = focus_per_sample.loc[
    [dataset.equilibrium_ids[row] for row in intermediate_rows], "iota_predicted_rmse"
].to_numpy()
# The highest-beta row whose prediction has a force residual below 1e-2. Every solve in
# the split is below it already, so the cut acts on the prediction alone.
test_predicted_residual = focus_per_sample.loc[
    [dataset.equilibrium_ids[row] for row in test_rows], "fnorm_predicted"
].to_numpy()
balanced_rows = test_rows[test_predicted_residual < 1e-2]
case_rows = (
    int(test_rows[test_beta.argmin()]),
    int(intermediate_rows[intermediate_iota_error.argmin()]),
    int(balanced_rows[beta_percent[balanced_rows].argmax()]),
)
# Each case is labelled by its volume-averaged beta, the quantity the three were picked by.
CASES = {
    r"$\langle \beta \rangle_{\mathrm{vol}} = "
    + f"{beta_percent[row]:.2f}"
    + r"\%$": row
    for row in case_rows
}
surrogate = model_storage_utils.load_surrogate(
    MODELS_DIR, FOCUS_MODEL, DATA_DIR, machine_scale=MACHINE_SCALE
)
# The stored rows are normalized: the pressure and the flat target are carried to
# the machine the way `read_desc_truth` carries a truth.
pressure = {
    label: np.asarray(dataset.pressure)[row] * MACHINE_SCALE.pressure_factor
    for label, row in CASES.items()
}
predicted = {
    label: surrogate_utils.unpack_flat_target(
        surrogate.basis, np.asarray(surrogate.predict(pressure[label]))
    )
    for label in CASES
}
target_scale_factors = surrogate_utils.flat_target_scale_factors(
    dataset.basis, MACHINE_SCALE
)
solved = {
    label: surrogate_utils.unpack_flat_target(
        dataset.basis, np.asarray(dataset.y)[row] * target_scale_factors
    )
    for label, row in CASES.items()
}
print(f"split median solve residual {np.median(solve_residual[test_rows]):.2e}")
for label, row in CASES.items():
    print(
        f"row {row:4d}  beta {beta_percent[row]:5.3f} percent  "
        f"p0 {pressure[label][0] * 1e-6:6.2f} MPa  "
        f"solve residual {solve_residual[row]:.2e}"
    )

The pressure each case is given (left) and the predicted transform, the network's `iotaf`
output, against the solve's (right). Section 2's `iota_computed_rmse` recomputes the
transform from the predicted geometry instead; if the two disagree, geometry and transform
were predicted inconsistently.

In [ ]:
# One colour per case, shared by every per-case figure below.
case_colours = {
    label: pubfig.CATEGORICAL_PALETTE[index % len(pubfig.CATEGORICAL_PALETTE)]
    for index, label in enumerate(CASES)
}
# Line widths of every per-case figure below, thick enough to read at print size.
LINE_WIDTH = 1.8
DASHED_LINE_WIDTH = 1.5


def subplots_with_legend_row():
    """Two panels side by side, with room above them for a legend row over both.

    The legend row adds its own height instead of taking it from the panels.
    """
    figure, axes = pubfig.subplots(1, 2, width="double", aspect=0.7)
    # A wider pad than the house style's, so the legend row does not sit on the panels.
    # Constrained layout spends the pad four times over the figure height, so the figure
    # grows by the same amount and the panels keep their shape.
    legend_gap_in = 0.12
    default_pad_in = plt.rcParams["figure.constrained_layout.h_pad"]
    figure.get_layout_engine().set(h_pad=legend_gap_in)
    figure.set_size_inches(
        figure.get_figwidth(),
        figure.get_figheight() + 0.3 + 4.0 * (legend_gap_in - default_pad_in),
    )
    return figure, axes


def case_legend_row(figure, case_colours):
    """One legend over both panels: colour is the case, highest beta first, and line
    style the model."""
    # Reversed here rather than in `CASES`, which would change each case's colour.
    figure.legend(
        handles=[
            plt.Line2D([], [], color=case_colours[label], lw=LINE_WIDTH, label=label)
            for label in reversed(CASES)
        ]
        + [
            plt.Line2D([], [], color="0.3", lw=LINE_WIDTH, label=PREDICTION_LABEL),
            plt.Line2D(
                [], [], color="0.3", lw=DASHED_LINE_WIDTH, ls="--", label=SOLVE_LABEL
            ),
        ],
        loc="outside upper center",
        ncol=5,
    )


# The pressure each case is given beside the transform predicted from it.
figure, axes = subplots_with_legend_row()
pressure_panel, iota_panel = axes
for label in CASES:
    pressure_panel.plot(
        np.asarray(surrogate.basis.pressure_rho_grid),
        1e-6 * pressure[label],
        color=case_colours[label],
        lw=LINE_WIDTH,
    )
    iota_panel.plot(
        np.asarray(surrogate.basis.iota_rho_grid),
        np.asarray(predicted[label].iotaf),
        color=case_colours[label],
        lw=LINE_WIDTH,
    )
    iota_panel.plot(
        np.asarray(surrogate.basis.iota_rho_grid),
        np.asarray(solved[label].iotaf),
        color=case_colours[label],
        ls="--",
        lw=DASHED_LINE_WIDTH,
    )
pressure_panel.set_xlabel(r"$\rho$")
pressure_panel.set_ylabel("pressure [MPa]")
iota_panel.set_xlabel(r"$\rho$")
iota_panel.set_ylabel(r"$\iota$")
# Steps of 0.005: the default 0.0025 needs four decimals on every label.
iota_panel.yaxis.set_major_locator(plt.MultipleLocator(0.005))
case_legend_row(figure, case_colours)
save_pdf(figure, "mhd_surrogate_profiles")

### Flux surfaces

Prediction and solve of the lowest- and highest-beta case, rebuilt in DESC at zero net
toroidal current under the solve's pressure spline, at four cross sections spanning half a
field period: the surfaces `rho = 0.25, 0.5, 0.75` and the boundary `rho = 1`, which is
fixed and so the same in every equilibrium, and the magnetic axis (dot: prediction, cross:
solve), where the shift is largest; the last panel enlarges the core of the `phi = pi/3`
section, where it shows best.
Prediction and solve nearly coincide, so read the legend: each entry carries the
equilibrium's normalized force residual, as in section 2. The outward displacement of the
high-beta surfaces is the Shafranov shift the pressure buys.

In [ ]:
# The spline the solver saw, not the sampled `pressure` column the surrogate is given.
splines = {
    label: desc_types.DescSplineProfile(
        name="pressure",
        knots=np.asarray(dataset.pressure_knots)[row],
        params=np.asarray(dataset.pressure_params)[row] * MACHINE_SCALE.pressure_factor,
    )
    for label, row in CASES.items()
}
predicted_equilibria = {
    label: surrogate_utils.desc_equilibrium_from_prediction(
        predicted[label], splines[label]
    )
    for label in CASES
}
solved_equilibria = {
    label: surrogate_utils.desc_equilibrium_from_prediction(
        solved[label], splines[label]
    )
    for label in CASES
}
# The two ends of the beta range only: the intermediate case's surfaces lie between them
# and crowd the shift the figure is there to show.
lowest_beta_case, *_, highest_beta_case = CASES
# Legend order: per case the prediction and the solve, so each legend column is one case.
equilibria = {}
colours = []
line_styles = []
axis_markers = []
for label in (lowest_beta_case, highest_beta_case):
    for reading, case_equilibria, line_style, axis_marker in (
        (PREDICTION_LABEL, predicted_equilibria, "-", "o"),
        (SOLVE_LABEL, solved_equilibria, "--", "x"),
    ):
        equilibria[f"{label}, {reading}"] = case_equilibria[label]
        colours.append(case_colours[label])
        line_styles.append(line_style)
        axis_markers.append(axis_marker)
residuals = {
    label: plots.normalized_force_residual(case, surrogate.basis.resolution)
    for label, case in equilibria.items()
}
for label, residual in residuals.items():
    print(f"{label:24s} residual {residual:.3e}")
# Half a field period, which stellarator symmetry mirrors onto the other half.
flux_surface_phi = np.linspace(
    0.0, np.pi / surrogate.basis.resolution.n_field_periods, 4
)
# The two tall sections span the full height side by side and the flat ones are stacked
# beside them, with a closer view of the last section's core under it: in a 2 x 2 grid
# the equal axis scale shrinks the tall ones to strips and leaves their row empty.
figure = plt.figure(figsize=(6.8, 6.0), layout="constrained")
panels = figure.subplot_mosaic(
    [
        ["phi_0", "phi_1", "phi_2"],
        ["phi_0", "phi_1", "phi_3"],
        ["phi_0", "phi_1", "core"],
    ]
)
axes = np.array([panels[f"phi_{index}"] for index in range(len(flux_surface_phi))])
core_panel = panels["core"]
# Out to the boundary, which is fixed and so the same in every equilibrium: the surfaces
# inside it shift against it.
flux_surface_rho = np.array([0.25, 0.5, 0.75, 1.0])
ideal_mhd_plot.plot_comparison(
    list(equilibria.values()),
    rho=flux_surface_rho,
    phi=flux_surface_phi,
    ax=axes,
    color=colours,
    ls=line_styles,
    labels=[f"{label}, fnorm {residual:.1e}" for label, residual in residuals.items()],
    legend_kw={"loc": "outside lower center", "ncol": 2},
)
ideal_mhd_plot.plot_comparison(
    list(equilibria.values()),
    rho=flux_surface_rho,
    phi=flux_surface_phi[-1:],
    ax=core_panel,
    color=colours,
    ls=line_styles,
    legend=False,
)
# The magnetic axis, where the shift is largest; `plot_comparison` hides it.
for panel, phi in zip([*axes, core_panel], [*flux_surface_phi, flux_surface_phi[-1]]):
    for equilibrium, colour, axis_marker in zip(
        equilibria.values(), colours, axis_markers
    ):
        axis_r, axis_z = desc_utils.evaluate_rz(
            equilibrium.coefficients, np.zeros(1), np.zeros(1), np.array([phi])
        )
        panel.scatter(
            np.asarray(axis_r),
            np.asarray(axis_z),
            color=colour,
            marker=axis_marker,
            s=22.0,
            zorder=5,
        )
    panel.xaxis.set_major_locator(plt.MaxNLocator(2))
# The core of the last section, centred between the axes: 12 cm by 7 cm at unit
# scale takes in the axis and the innermost surface, whose shift is a few
# millimetres there.
core_half_width = 0.06 * MACHINE_SCALE.length_factor
core_half_height = 0.035 * MACHINE_SCALE.length_factor
core_r, core_z = np.mean(
    [
        np.concatenate(
            desc_utils.evaluate_rz(
                equilibrium.coefficients,
                np.zeros(1),
                np.zeros(1),
                flux_surface_phi[-1:],
            )
        )
        for equilibrium in equilibria.values()
    ],
    axis=0,
)
core_panel.set_xlim(core_r - core_half_width, core_r + core_half_width)
core_panel.set_ylim(core_z - core_half_height, core_z + core_half_height)
core_panel.set_title(r"core of $\phi = 0.33\pi$")
axes[-1].indicate_inset(
    (
        core_r - core_half_width,
        core_z - core_half_height,
        2.0 * core_half_width,
        2.0 * core_half_height,
    ),
    inset_ax=core_panel,
    edgecolor="0.3",
)
# Each column as wide as its panels are at the height they get, the full height for the
# first two and a third of it for the stacked three, so no panel is padded with space.
panel_shape = [
    np.ptp(panel.get_xlim()) / np.ptp(panel.get_ylim()) for panel in [*axes, core_panel]
]
axes[0].get_gridspec().set_width_ratios(
    [panel_shape[0], panel_shape[1], max(panel_shape[2:]) / 3.0]
)
# Vertical `R` labels on the phi = 0 section only, the narrowest.
axes[0].tick_params(axis="x", labelrotation=90.0)
figure.suptitle(
    r"flux surfaces $\rho = 0.25$, $0.5$, $0.75$, $1$ and the magnetic axis"
)
save_pdf(figure, "mhd_surrogate_flux_surfaces_vs_vacuum")

### Magnetic well and Mercier criterion

Surface averages of second derivatives of the geometry, so the harshest test of a prediction:
matching flux surfaces need not mean matching curvature. The well changes sign, so it is drawn
linearly and its zero separates a well from a hill; `D_Mercier` is negative everywhere here, so
its magnitude is on a log axis. Solid is the prediction, dashed the solve; a gap between lines
of one colour is a disagreement about stability.

In [ ]:
resolution = surrogate.basis.resolution
# Neither quantity is defined on the magnetic axis, so the innermost of the 32 surfaces
# sits one step inside the plasma rather than at the axis.
stability_grid = desc.grid.LinearGrid(
    rho=np.linspace(1.0, 0.0, 32, endpoint=False)[::-1],
    M=resolution.poloidal_resolution,
    N=resolution.toroidal_resolution,
    NFP=resolution.n_field_periods,
    sym=resolution.is_stellarator_symmetric,
)
stability_rho = pd.Index(
    np.asarray(stability_grid.compress(stability_grid.nodes[:, 0])), name="rho"
)
predicted_stability = {}
solved_stability = {}
for stability, case_equilibria in (
    (predicted_stability, predicted_equilibria),
    (solved_stability, solved_equilibria),
):
    for label, equilibrium in case_equilibria.items():
        data = desc_utils.desc_equilibrium_to_desc_object(
            equilibrium, resolution
        ).compute(["magnetic well", "D_Mercier"], grid=stability_grid)
        stability[label] = pd.DataFrame(
            {
                "magnetic_well": np.asarray(
                    stability_grid.compress(data["magnetic well"])
                ),
                "d_mercier": np.asarray(stability_grid.compress(data["D_Mercier"])),
            },
            index=stability_rho,
        )
figure, axes = subplots_with_legend_row()
for label in CASES:
    predicted_profiles = predicted_stability[label]
    solved_profiles = solved_stability[label]
    axes[0].plot(
        predicted_profiles.index,
        predicted_profiles["magnetic_well"],
        color=case_colours[label],
        lw=LINE_WIDTH,
        label=f"{label}, {PREDICTION_LABEL}",
    )
    axes[1].plot(
        predicted_profiles.index,
        -predicted_profiles["d_mercier"],
        color=case_colours[label],
        lw=LINE_WIDTH,
        label=f"{label}, {PREDICTION_LABEL}",
    )
    axes[0].plot(
        solved_profiles.index,
        solved_profiles["magnetic_well"],
        color=case_colours[label],
        ls="--",
        lw=DASHED_LINE_WIDTH,
        label=f"{label}, {SOLVE_LABEL}",
    )
    axes[1].plot(
        solved_profiles.index,
        -solved_profiles["d_mercier"],
        color=case_colours[label],
        ls="--",
        lw=DASHED_LINE_WIDTH,
        label=f"{label}, {SOLVE_LABEL}",
    )
axes[0].axhline(0.0, color="0.6", lw=0.6)
axes[0].set_xlabel(r"$\rho$")
axes[0].set_ylabel("magnetic well")
axes[1].set_xlabel(r"$\rho$")
axes[1].set_ylabel(r"$-D_{\mathrm{Mercier}}$ [Wb$^{-2}$]")
axes[1].set_yscale("log")
case_legend_row(figure, case_colours)
save_pdf(figure, "mhd_surrogate_stability_profiles")
for label in CASES:
    print(
        f"{label:12s} magnetic well, largest value: predicted "
        f"{predicted_stability[label]['magnetic_well'].max():+.3e}, solved "
        f"{solved_stability[label]['magnetic_well'].max():+.3e}"
    )

### The field strength

The prediction's `|B|` error relative to the solve's local `|B|`, in percent: first its mean
over each flux surface against radius, with the interquartile range over the surface
shaded (after Merlo et al. 2023, Nucl. Fusion 63 066020, Fig. 9), then
where on the surface it sits and with which sign, on mid-radius and the boundary (after
Merlo et al. 2021, Fig. 15). `|B|` is rebuilt from each equilibrium's Boozer harmonics on
one field period, so both are compared at the same Boozer angles; the contours are the
solve's `|B|`, on which quasi-isodynamicity, the property the configuration was optimized
for, is a condition. The map's colour scale is logarithmic on both sides of zero and
linear within 0.01 %, so the two low-beta cases are not blank beside the high-beta one.
The model was never trained on `|B|`, only on a coefficient vector.

In [ ]:
# The 32 surfaces of the stability figure: the Boozer angles are not defined on the
# axis, where the poloidal angle degenerates.
boozer_rho = stability_rho.to_numpy()
boozer_spectrum = {}
for reading, case_equilibria in (
    (PREDICTION_LABEL, predicted_equilibria),
    (SOLVE_LABEL, solved_equilibria),
):
    for label, equilibrium in case_equilibria.items():
        desc_object = desc_utils.desc_equilibrium_to_desc_object(
            equilibrium, resolution
        )
        # The resolution of `desc.plotting.plot_boozer_modes`: Boozer modes to twice the
        # equilibrium's, on a grid fine enough to resolve them.
        boozer_grid = desc.grid.LinearGrid(
            M=2 * desc_object.M_grid,
            N=2 * desc_object.N_grid,
            NFP=desc_object.NFP,
            rho=boozer_rho,
        )
        transforms = desc.compute.get_transforms(
            "|B|_mn_B",
            obj=desc_object,
            grid=boozer_grid,
            M_booz=2 * desc_object.M,
            N_booz=2 * desc_object.N,
        )
        coefficients = np.asarray(
            desc_object.compute("|B|_mn_B", grid=boozer_grid, transforms=transforms)[
                "|B|_mn_B"
            ]
        ).reshape(len(boozer_rho), -1)
        # DESC's double Fourier series to the `B_mn cos(m theta - n NFP zeta)` form the
        # literature quotes.
        matrix, modes = desc.vmec_utils.ptolemy_linear_transform(
            transforms["B"].basis.modes
        )
        assert (modes[:, 0] == 1).all(), "a stellarator-symmetric |B| has cosines only"
        boozer_spectrum[label, reading] = pd.DataFrame(
            (matrix @ coefficients.T).T,
            index=pd.Index(boozer_rho, name="rho"),
            columns=pd.MultiIndex.from_arrays(
                [modes[:, 1], modes[:, 2]], names=["m", "n"]
            ),
        )
# `|B|` on every surface, rebuilt from its harmonics on one field period in Boozer
# angles (fractions of a turn), so prediction and solve are compared at the same point.
theta_fraction = np.linspace(0.0, 1.0, 64, endpoint=False)
zeta_fraction = np.linspace(0.0, 1.0, 64, endpoint=False)
harmonics = boozer_spectrum[next(iter(CASES)), SOLVE_LABEL].columns
boozer_basis = np.cos(
    2.0
    * np.pi
    * (
        harmonics.get_level_values("m").to_numpy()[:, None, None]
        * theta_fraction[None, :, None]
        - harmonics.get_level_values("n").to_numpy()[:, None, None]
        * zeta_fraction[None, None, :]
    )
)
field_on_surfaces = {
    key: np.tensordot(spectrum.to_numpy(), boozer_basis, axes=1).reshape(
        len(boozer_rho), -1
    )
    for key, spectrum in boozer_spectrum.items()
}
# Relative to the local field, in percent, a scale the reader needs no reference
# field for.
field_error_percent = {
    label: 100.0
    * np.abs(
        field_on_surfaces[label, PREDICTION_LABEL]
        - field_on_surfaces[label, SOLVE_LABEL]
    )
    / field_on_surfaces[label, SOLVE_LABEL]
    for label in CASES
}
# Beside the maps in the thesis, at 0.47 of the text width and as tall as them: drawn
# at 2.9 in, it prints at about 0.95 of its size.
figure, axes = pubfig.subplots(width=2.9, aspect=1.15)
panel = axes[0]
for label in CASES:
    error = field_error_percent[label]
    # The spread over each surface as its interquartile range: a standard deviation would
    # reach below zero on the log axis, and the 5 to 95 % bands of the three cases
    # overlap into one.
    low, high = np.percentile(error, [25.0, 75.0], axis=1)
    panel.fill_between(
        boozer_rho, low, high, color=case_colours[label], alpha=0.2, lw=0.0
    )
    panel.plot(boozer_rho, error.mean(axis=1), color=case_colours[label], lw=LINE_WIDTH)
panel.set_yscale("log")
panel.set_xlabel(r"$\rho$")
panel.set_ylabel(f"{PREDICTION_LABEL} $|B|$ error [%]")
# Above the axes, the cases only: inside, the legend covers a curve wherever it is put,
# and the caption says what the line and the band are.
figure.legend(
    handles=[
        plt.Line2D(
            [],
            [],
            color=colour,
            lw=LINE_WIDTH,
            label=label.replace(r"_{\mathrm{vol}}", ""),
        )
        for label, colour in case_colours.items()
    ],
    loc="outside upper center",
    ncol=1,
)
save_pdf(figure, "mhd_surrogate_field_strength_error")
for label in CASES:
    error = field_error_percent[label]
    rms = np.sqrt(np.mean(error**2, axis=1))
    print(
        f"{label:12s} RMS {rms[boozer_rho == 0.5][0]:.3f} % at rho 0.5, "
        f"{rms[-1]:.3f} % at rho 1; largest {error.max():.3f} % at rho "
        f"{boozer_rho[error.max(axis=1).argmax()]:.2f}"
    )

# Where on the surface the error sits, and its sign, on mid-radius and the boundary.
map_rho = (0.5, 1.0)
signed_error_percent = {
    (label, rho): (
        100.0
        * (
            field_on_surfaces[label, PREDICTION_LABEL]
            - field_on_surfaces[label, SOLVE_LABEL]
        )
        / field_on_surfaces[label, SOLVE_LABEL]
    )[boozer_rho == rho][0].reshape(len(theta_fraction), len(zeta_fraction))
    for label in CASES
    for rho in map_rho
}
# Logarithmic in the magnitude on both sides of zero and linear below a hundredth of a
# percent: a linear scale shared by the three cases leaves the two smaller ones blank.
error_limit = max(np.abs(error).max() for error in signed_error_percent.values())
error_norm = mpl_colors.SymLogNorm(
    linthresh=1e-2, vmin=-error_limit, vmax=error_limit, base=10
)
# Beside the profile in the thesis, at 0.52 of the text width: drawn at 3.2 in, it prints
# at about 0.95 of its size. One row per case and one column per surface.
figure, axes = pubfig.subplots(
    len(CASES), len(map_rho), width=3.2, aspect=0.7, sharex=True, sharey=True
)
for row, label in enumerate(CASES):
    for column, rho in enumerate(map_rho):
        panel = axes[row, column]
        mesh = panel.pcolormesh(
            zeta_fraction,
            theta_fraction,
            signed_error_percent[label, rho],
            cmap="RdBu_r",
            norm=error_norm,
            shading="nearest",
            rasterized=True,
        )
        # The field the error sits on: contours of the DESC `|B|`, kept faint.
        panel.contour(
            zeta_fraction,
            theta_fraction,
            field_on_surfaces[label, SOLVE_LABEL][boozer_rho == rho][0].reshape(
                len(theta_fraction), len(zeta_fraction)
            ),
            levels=6,
            colors="0.35",
            linewidths=0.5,
        )
    axes[row, 0].set_ylabel(r"$\theta_{\mathrm{B}} / 2\pi$")
    # The case on the right of its row, in its colour in the profile beside it.
    axes[row, -1].yaxis.set_label_position("right")
    axes[row, -1].set_ylabel(
        label.replace(r"_{\mathrm{vol}}", ""),
        color=case_colours[label],
        fontsize="small",
    )
for column, rho in enumerate(map_rho):
    axes[0, column].set_title(rf"$\rho = {rho:g}$")
# One axis label for the two columns, which share it.
figure.supxlabel(
    r"$N_{\mathrm{FP}} \zeta_{\mathrm{B}} / 2\pi$",
    fontsize=plt.rcParams["axes.labelsize"],
)
colour_bar = figure.colorbar(
    mesh, ax=axes, label=f"{PREDICTION_LABEL} $-$ DESC [%]", shrink=0.9
)
colour_bar.set_ticks(
    [-0.1, -0.01, 0.0, 0.01, 0.1], labels=["-0.1", "-0.01", "0", "0.01", "0.1"]
)
save_pdf(figure, "mhd_surrogate_field_strength_error_maps")

### The global quantities the predictions imply

The `wout` scalars of a VMEC run, computed with DESC from each geometry under the solve's
pressure spline, for the prediction and the solve of each case. They follow from geometry and
pressure alone, so none echoes the predicted transform.

In [ ]:
table = plots.global_scalar_table(
    {
        **predicted_equilibria,
        **{f"{label}, solve": case for label, case in solved_equilibria.items()},
    },
    surrogate.basis.resolution,
)
table.to_csv(TABLES_DIR / "mhd_surrogate_global_scalars.csv")
table

### 3.2 Force balance against plasma beta

The whole test split, binned by the volume-averaged beta of its DESC solve (one bin per percent
up to five, everything above in the last). Per bin: the mean force residual of the predictions
and of the solves, with the standard error of the mean, `std / sqrt(N)`. The solves' bar
separates the model getting worse from the equilibria getting harder to solve. The lower panel
counts the **training** equilibria per bin: beta was never sampled directly, and the bins where
the model is worst are the ones it saw least of.

In [ ]:
per_sample = pd.read_csv(PER_SAMPLE_CSV)
assert (per_sample["model"] == FOCUS_MODEL).any(), (
    f"{FOCUS_MODEL} is not in {PER_SAMPLE_CSV.name}; name it in "
    "run_test_set_evaluation.py --models"
)
focus_rows = per_sample[per_sample["model"] == FOCUS_MODEL]
# A bin includes its lower edge and excludes its upper one.
BETA_BIN_EDGES = (0.0, 1.0, 2.0, 3.0, 4.0, 5.0, np.inf)
bin_edges = list(zip(BETA_BIN_EDGES[:-1], BETA_BIN_EDGES[1:], strict=True))
bin_labels = pd.Index(
    [
        f">{low:.0f}%" if np.isinf(high) else f"{low:.0f}-{high:.0f}%"
        for low, high in bin_edges
    ],
    name="beta_vol",
)
# The bin of every dataset row, by equilibrium id, the key the per-sample table uses.
dataset_bins = pd.Series(
    np.asarray(
        pd.cut(
            beta_percent,
            bins=list(BETA_BIN_EDGES),
            right=False,
            labels=list(bin_labels),
        )
    ),
    index=dataset.equilibrium_ids,
)
assert focus_rows["equilibrium_id"].isin(dataset_bins.index).all(), (
    "some scored equilibria are not in the dataset; "
    "the table and the dataset are of different campaigns"
)
sample_bins = dataset_bins.reindex(focus_rows["equilibrium_id"].to_numpy()).to_numpy()
binned_rows = []
for label, (low, high) in zip(bin_labels, bin_edges, strict=True):
    in_bin = sample_bins == label
    binned_row = {"beta_low": low, "beta_high": high, "n_samples": int(in_bin.sum())}
    for side in ("predicted", "target"):
        values = focus_rows.loc[in_bin, f"fnorm_{side}"].to_numpy(float)
        # `nan` where a bin is too small for the statistic: a mean needs one value,
        # the sample standard deviation two.
        binned_row[f"{side}_mean"] = (
            float(np.mean(values)) if len(values) else float("nan")
        )
        binned_row[f"{side}_standard_error"] = (
            float(np.std(values, ddof=1) / np.sqrt(len(values)))
            if len(values) > 1
            else float("nan")
        )
    binned_rows.append(binned_row)
binned = pd.DataFrame(binned_rows, index=bin_labels)
train_counts = (
    pd.Series(dataset_bins.to_numpy()[np.asarray(split.train)])
    .value_counts()
    .reindex(bin_labels, fill_value=0)
)
table = binned.assign(n_train=train_counts)
table.to_csv(TABLES_DIR / "mhd_surrogate_force_residual_by_beta.csv")

figure, axes = pubfig.subplots(2, 1, width=4.8, aspect=0.45, sharex=True)
positions = np.arange(len(binned), dtype=float)
beta_bar_width = 0.76 / 2
for offset, (side, label, colour) in enumerate(
    (("predicted", PREDICTION_LABEL, 0), ("target", SOLVE_LABEL, 1))
):
    axes[0].bar(
        positions + (offset - 0.5) * beta_bar_width,
        binned[f"{side}_mean"],
        width=beta_bar_width,
        yerr=binned[f"{side}_standard_error"],
        color=pubfig.CATEGORICAL_PALETTE[colour],
        label=label,
    )
axes[0].set_ylabel("force residual")
axes[0].set_yscale("log")
# Inside the axes on a translucent frame, so the panel keeps the full width.
axes[0].legend(loc="best", frameon=True, framealpha=0.8, edgecolor="none")
axes[1].bar(positions, train_counts.to_numpy(), width=2.0 * beta_bar_width, color="0.6")
axes[1].set_ylabel("training equilibria")
# Headroom for the count above each bar, which otherwise sits on the top spine.
axes[1].set_ylim(top=1.18 * float(train_counts.max()))
# No unit on the axis label: every tick carries its own percent sign.
axes[1].set_xlabel(r"$\langle \beta \rangle_{\mathrm{vol}}$")
axes[1].set_xticks(positions)
# cmr10 has no `>` glyph (matplotlib substitutes an inverted question mark), so the
# open top bin is labelled in mathtext.
axes[1].set_xticklabels(
    [
        rf"$>\!{low:.0f}\%$" if np.isinf(high) else label
        for label, (low, high) in zip(bin_labels, bin_edges, strict=True)
    ]
)
for position, count in zip(positions, train_counts.to_numpy(), strict=True):
    axes[1].annotate(
        str(int(count)),
        (position, count),
        textcoords="offset points",
        xytext=(0.0, 2.0),
        ha="center",
        va="bottom",
        fontsize=plt.rcParams["font.size"] - 3.0,
    )
save_pdf(figure, "mhd_surrogate_force_residual_by_beta")
table

## 4. The equilibrium interpolator

Trilinear interpolation of the flat target over the solved `(p0, a, b)` lattice in
`data/equilibrium_a_interpolator_28_spline_input.parquet`. Section 2 scores it; this section shows
the two lattice properties that make it the floor rather than a competitor, in the
lattice's own normalized units, as stored.

### 4.1 Exactness on the lattice

At a lattice node it returns that node's solved equilibrium exactly.

In [ ]:
INTERPOLATOR = "equilibrium_interpolator"
assert (MODELS_DIR / f"{INTERPOLATOR}.json").exists(), (
    f"no {INTERPOLATOR} entry in {MODELS_DIR}; write one with "
    "model_storage_utils.save_interpolator_model"
)
stored = model_storage_utils.load_model_metadata(MODELS_DIR, INTERPOLATOR)
lattice = interpolator_utils.load_lattice(
    DATA_DIR / stored["provenance"]["lattice_parquet"],
    DATA_DIR / stored["provenance"]["lattice_settings"],
)
surrogate = model_storage_utils.load_surrogate(MODELS_DIR, INTERPOLATOR, DATA_DIR)
node = (len(lattice.p0_grid) - 1, 1, 0)
at_node = interpolator_utils.predict_y_from_params(
    lattice,
    preprocessing_types.PressureParams(
        p0=float(lattice.p0_grid[node[0]]),
        a=float(lattice.a_grid[node[1]]),
        b=float(lattice.b_grid[node[2]]),
    ),
)
np.testing.assert_allclose(
    np.asarray(at_node), np.asarray(lattice.y_lattice)[node], rtol=0, atol=1e-12
)
print(
    f"lattice {np.asarray(lattice.y_lattice).shape}: "
    f"p0 {np.asarray(lattice.p0_grid) * 1e-3} kPa, a {np.asarray(lattice.a_grid)}, "
    f"b {np.asarray(lattice.b_grid)}"
)
print(f"node {node} recovered exactly")

### 4.2 Reach of the lattice

A query outside the `(p0, a, b)` box silently saturates to the boundary node (the
interpolation weight is clipped into `[0, 1]`). Most profiles were drawn from a Gaussian
process, so each is first fitted at its closest representable shape; this is the fraction of
test profiles whose fit lands outside the box.

In [ ]:
# The split loaded in section 3.1, so the fraction is quoted over the same test rows the
# comparison table was scored on.
test_pressure = np.asarray(dataset.pressure)[test_rows]
outside = interpolator_utils.fraction_outside_lattice(lattice, test_pressure)
print(f"{outside:.1%} of the {len(test_pressure)} test profiles are outside the box")